# 🎓 Smart School Management System — Phase 1
**Project setup, database and login system** (runs fully inside Google Colab)

### How to use this notebook
1. Click **Runtime → Run all** (or press the ▶ button on each cell, top to bottom).
2. Wait for the last cell. It prints a **public link** (ends with `trycloudflare.com`). Click it.
3. Log in with a demo account:

| Role | Username | Password |
|---|---|---|
| Admin | `admin` | `admin123` |
| Teacher | `teacher1` | `teacher123` |
| Student | `student1` | `student123` |

**Important:** Colab deletes files when the session ends. Just run all cells again next time. The link changes every run.


## Step 1 — Install and create folders

In [ ]:
!pip -q install flask werkzeug
!mkdir -p school_sms/templates school_sms/static/css school_sms/static/js
print("✅ Folders ready")

In [ ]:
%%writefile school_sms/requirements.txt
flask
werkzeug
openai


## Step 2 — Database (all tables + demo data)

In [ ]:
%%writefile school_sms/database.py
import os
import sqlite3
from datetime import date, timedelta
from werkzeug.security import generate_password_hash

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DB_PATH = os.path.join(BASE_DIR, "school.db")


def get_db():
    """Open a connection to the SQLite database."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


SCHEMA = """
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password_hash TEXT NOT NULL,
    role TEXT NOT NULL CHECK (role IN ('admin','teacher','student')),
    full_name TEXT NOT NULL,
    email TEXT
);

CREATE TABLE IF NOT EXISTS classes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE NOT NULL,
    class_teacher_id INTEGER,
    FOREIGN KEY (class_teacher_id) REFERENCES teachers(id) ON DELETE SET NULL
);

CREATE TABLE IF NOT EXISTS sections (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    class_id INTEGER NOT NULL,
    name TEXT NOT NULL,
    FOREIGN KEY (class_id) REFERENCES classes(id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS subjects (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE NOT NULL,
    code TEXT UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS teachers (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER UNIQUE NOT NULL,
    employee_id TEXT UNIQUE NOT NULL,
    phone TEXT,
    qualification TEXT,
    joining_date TEXT,
    FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS teacher_assignments (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    teacher_id INTEGER NOT NULL,
    class_id INTEGER NOT NULL,
    section_id INTEGER,
    subject_id INTEGER NOT NULL,
    FOREIGN KEY (teacher_id) REFERENCES teachers(id) ON DELETE CASCADE,
    FOREIGN KEY (class_id) REFERENCES classes(id) ON DELETE CASCADE,
    FOREIGN KEY (section_id) REFERENCES sections(id) ON DELETE CASCADE,
    FOREIGN KEY (subject_id) REFERENCES subjects(id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER UNIQUE,
    admission_no TEXT UNIQUE NOT NULL,
    full_name TEXT NOT NULL,
    class_id INTEGER,
    section_id INTEGER,
    dob TEXT,
    gender TEXT,
    guardian_name TEXT,
    phone TEXT,
    address TEXT,
    FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE SET NULL,
    FOREIGN KEY (class_id) REFERENCES classes(id) ON DELETE SET NULL,
    FOREIGN KEY (section_id) REFERENCES sections(id) ON DELETE SET NULL
);

CREATE TABLE IF NOT EXISTS attendance (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    date TEXT NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('Present','Absent')),
    marked_by INTEGER,
    UNIQUE (student_id, date),
    FOREIGN KEY (student_id) REFERENCES students(id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS examinations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    class_id INTEGER NOT NULL,
    start_date TEXT,
    end_date TEXT,
    FOREIGN KEY (class_id) REFERENCES classes(id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS marks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    exam_id INTEGER NOT NULL,
    student_id INTEGER NOT NULL,
    subject_id INTEGER NOT NULL,
    marks_obtained REAL NOT NULL,
    max_marks REAL NOT NULL DEFAULT 100,
    UNIQUE (exam_id, student_id, subject_id),
    FOREIGN KEY (exam_id) REFERENCES examinations(id) ON DELETE CASCADE,
    FOREIGN KEY (student_id) REFERENCES students(id) ON DELETE CASCADE,
    FOREIGN KEY (subject_id) REFERENCES subjects(id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS notices (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    content TEXT NOT NULL,
    created_at TEXT NOT NULL,
    created_by INTEGER,
    FOREIGN KEY (created_by) REFERENCES users(id) ON DELETE SET NULL
);

CREATE TABLE IF NOT EXISTS school_info (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    topic TEXT UNIQUE NOT NULL,
    content TEXT NOT NULL
);
"""


def init_db():
    """Create tables and add demo data (only the first time)."""
    conn = get_db()
    conn.executescript(SCHEMA)

    already = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
    if already:
        conn.close()
        return

    today = date.today()
    d = lambda n: (today + timedelta(days=n)).isoformat()

    # --- Users (passwords are stored hashed, never as plain text)
    users = [
        ("admin", "admin123", "admin", "Principal Sharma", "admin@school.edu"),
        ("teacher1", "teacher123", "teacher", "Anita Verma", "anita@school.edu"),
        ("teacher2", "teacher123", "teacher", "Rahul Mehta", "rahul@school.edu"),
        ("student1", "student123", "student", "Aarav Singh", "aarav@school.edu"),
        ("student2", "student123", "student", "Diya Patel", "diya@school.edu"),
    ]
    ids = {}
    for u, p, role, name, email in users:
        cur = conn.execute(
            "INSERT INTO users (username, password_hash, role, full_name, email) VALUES (?,?,?,?,?)",
            (u, generate_password_hash(p), role, name, email))
        ids[u] = cur.lastrowid

    # --- Teachers
    conn.execute("INSERT INTO teachers (user_id, employee_id, phone, qualification, joining_date) VALUES (?,?,?,?,?)",
                 (ids["teacher1"], "T001", "9876500001", "M.Sc Mathematics, B.Ed", "2018-06-01"))
    conn.execute("INSERT INTO teachers (user_id, employee_id, phone, qualification, joining_date) VALUES (?,?,?,?,?)",
                 (ids["teacher2"], "T002", "9876500002", "M.A English, B.Ed", "2020-07-15"))

    # --- Classes, sections, subjects
    for c in ("Class 9", "Class 10"):
        cur = conn.execute("INSERT INTO classes (name) VALUES (?)", (c,))
        for s in ("A", "B"):
            conn.execute("INSERT INTO sections (class_id, name) VALUES (?,?)", (cur.lastrowid, s))
    conn.execute("UPDATE classes SET class_teacher_id = 1 WHERE name = 'Class 9'")
    conn.execute("UPDATE classes SET class_teacher_id = 2 WHERE name = 'Class 10'")

    for name, code in [("Mathematics", "MATH"), ("English", "ENG"), ("Science", "SCI"),
                       ("Social Studies", "SST"), ("Computer Science", "CS")]:
        conn.execute("INSERT INTO subjects (name, code) VALUES (?,?)", (name, code))

    conn.execute("INSERT INTO teacher_assignments (teacher_id, class_id, section_id, subject_id) VALUES (1,1,1,1)")
    conn.execute("INSERT INTO teacher_assignments (teacher_id, class_id, section_id, subject_id) VALUES (1,2,1,1)")
    conn.execute("INSERT INTO teacher_assignments (teacher_id, class_id, section_id, subject_id) VALUES (2,1,1,2)")
    conn.execute("INSERT INTO teacher_assignments (teacher_id, class_id, section_id, subject_id) VALUES (2,2,2,2)")

    # --- Students
    students = [
        (ids["student1"], "ADM001", "Aarav Singh", 1, 1, "2010-03-14", "Male", "Rajesh Singh", "9876510001", "12 Park Street, Delhi"),
        (ids["student2"], "ADM002", "Diya Patel", 1, 1, "2010-07-22", "Female", "Meena Patel", "9876510002", "45 Lake Road, Delhi"),
        (None, "ADM003", "Kabir Khan", 1, 2, "2010-01-30", "Male", "Imran Khan", "9876510003", "8 Green Avenue, Delhi"),
        (None, "ADM004", "Sara Joseph", 2, 3, "2009-11-05", "Female", "Thomas Joseph", "9876510004", "90 Hill View, Delhi"),
        (None, "ADM005", "Vihaan Gupta", 2, 3, "2009-09-18", "Male", "Sunita Gupta", "9876510005", "23 Main Market, Delhi"),
        (None, "ADM006", "Ananya Rao", 2, 4, "2009-12-02", "Female", "Prakash Rao", "9876510006", "67 Sunrise Colony, Delhi"),
    ]
    for s in students:
        conn.execute("""INSERT INTO students
            (user_id, admission_no, full_name, class_id, section_id, dob, gender, guardian_name, phone, address)
            VALUES (?,?,?,?,?,?,?,?,?,?)""", s)

    # --- Attendance: last 5 days (skipping nothing, just demo data)
    for back in range(0, 5):
        for sid in range(1, 7):
            status = "Absent" if (sid + back) % 7 == 0 else "Present"
            conn.execute("INSERT INTO attendance (student_id, date, status, marked_by) VALUES (?,?,?,?)",
                         (sid, d(-back), status, ids["teacher1"]))

    # --- Exams
    conn.execute("INSERT INTO examinations (name, class_id, start_date, end_date) VALUES (?,?,?,?)",
                 ("Mid-Term Examination", 1, d(14), d(21)))
    conn.execute("INSERT INTO examinations (name, class_id, start_date, end_date) VALUES (?,?,?,?)",
                 ("Mid-Term Examination", 2, d(15), d(22)))

    # --- Notices
    conn.execute("INSERT INTO notices (title, content, created_at, created_by) VALUES (?,?,?,?)",
                 ("Welcome to the New Term", "We welcome all students and parents to the new academic term.", d(-3), ids["admin"]))
    conn.execute("INSERT INTO notices (title, content, created_at, created_by) VALUES (?,?,?,?)",
                 ("Parent-Teacher Meeting", "The parent-teacher meeting will be held on the first Saturday of next month.", d(-1), ids["admin"]))
    conn.execute("INSERT INTO notices (title, content, created_at, created_by) VALUES (?,?,?,?)",
                 ("Sports Day Practice", "Sports day practice starts next Monday after school hours.", d(0), ids["admin"]))

    # --- School information (used by the AI chatbot in Phase 8)
    info = [
        ("School timings", "School runs Monday to Friday from 8:00 AM to 2:30 PM. Saturday is a half day, 8:00 AM to 12:00 PM."),
        ("School rules", "Students must wear the proper uniform, arrive on time, and respect teachers and classmates. Mobile phones are not allowed in class."),
        ("Holidays", "The school is closed on Sundays and national holidays. Summer vacation is in June and winter break is in late December."),
        ("Facilities", "Library, science and computer labs, sports ground, playground, auditorium and canteen."),
        ("Admission procedure", "Collect the admission form from the office, submit it with the birth certificate and previous report card, then attend a short interaction."),
    ]
    for topic, content in info:
        conn.execute("INSERT INTO school_info (topic, content) VALUES (?,?)", (topic, content))

    conn.commit()
    conn.close()
    print("Database created with demo data.")


if __name__ == "__main__":
    init_db()


## Step 3 — Flask application (login + role protection)

In [ ]:
%%writefile school_sms/app.py
import os
from datetime import date
from functools import wraps

from flask import (Flask, render_template, request, redirect,
                   url_for, session, flash)
from werkzeug.security import check_password_hash

from database import get_db, init_db

app = Flask(__name__)
# In real life, set a SECRET_KEY environment variable. This default is for the demo.
app.secret_key = os.environ.get("SECRET_KEY", "change-this-secret-key-for-demo")

init_db()


# ------------------------------------------------------------ security helpers
def login_required(view):
    """Only logged-in users can open the page."""
    @wraps(view)
    def wrapper(*args, **kwargs):
        if "user_id" not in session:
            flash("Please log in first.", "error")
            return redirect(url_for("login"))
        return view(*args, **kwargs)
    return wrapper


def role_required(*roles):
    """Only users with one of the given roles can open the page."""
    def decorator(view):
        @wraps(view)
        def wrapper(*args, **kwargs):
            if session.get("role") not in roles:
                flash("You do not have permission to open that page.", "error")
                return redirect(url_for("dashboard"))
            return view(*args, **kwargs)
        return wrapper
    return decorator


# ------------------------------------------------------------------- routes
@app.route("/")
def index():
    if "user_id" in session:
        return redirect(url_for("dashboard"))
    return redirect(url_for("login"))


@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form.get("username", "").strip()
        password = request.form.get("password", "")
        role = request.form.get("role", "admin")

        db = get_db()
        user = db.execute("SELECT * FROM users WHERE username = ? AND role = ?",
                          (username, role)).fetchone()
        db.close()

        if user and check_password_hash(user["password_hash"], password):
            session.clear()
            session["user_id"] = user["id"]
            session["username"] = user["username"]
            session["role"] = user["role"]
            session["full_name"] = user["full_name"]
            flash("Welcome back, " + user["full_name"] + "!", "success")
            return redirect(url_for("dashboard"))

        flash("Wrong username, password or role. Please try again.", "error")
    return render_template("login.html")


@app.route("/logout")
def logout():
    session.clear()
    flash("You have been logged out.", "success")
    return redirect(url_for("login"))


@app.route("/dashboard")
@login_required
def dashboard():
    db = get_db()
    today = date.today().isoformat()
    role = session["role"]
    data = {}

    data["notices"] = db.execute(
        "SELECT * FROM notices ORDER BY created_at DESC, id DESC LIMIT 5").fetchall()

    if role == "admin":
        data["total_students"] = db.execute("SELECT COUNT(*) FROM students").fetchone()[0]
        data["total_teachers"] = db.execute("SELECT COUNT(*) FROM teachers").fetchone()[0]
        data["total_classes"] = db.execute("SELECT COUNT(*) FROM classes").fetchone()[0]
        data["present_today"] = db.execute(
            "SELECT COUNT(*) FROM attendance WHERE date = ? AND status = 'Present'", (today,)).fetchone()[0]
        data["marked_today"] = db.execute(
            "SELECT COUNT(*) FROM attendance WHERE date = ?", (today,)).fetchone()[0]
        data["exams"] = db.execute(
            """SELECT e.name, e.start_date, e.end_date, c.name AS class_name
               FROM examinations e JOIN classes c ON c.id = e.class_id
               WHERE e.start_date >= ? ORDER BY e.start_date LIMIT 5""", (today,)).fetchall()

    elif role == "teacher":
        teacher = db.execute(
            """SELECT t.*, u.full_name, u.email FROM teachers t
               JOIN users u ON u.id = t.user_id WHERE t.user_id = ?""",
            (session["user_id"],)).fetchone()
        data["teacher"] = teacher
        data["assignments"] = db.execute(
            """SELECT c.name AS class_name, s.name AS section_name, sub.name AS subject_name
               FROM teacher_assignments ta
               JOIN classes c ON c.id = ta.class_id
               LEFT JOIN sections s ON s.id = ta.section_id
               JOIN subjects sub ON sub.id = ta.subject_id
               WHERE ta.teacher_id = ?""", (teacher["id"],)).fetchall()

    else:  # student
        student = db.execute(
            """SELECT st.*, c.name AS class_name, sec.name AS section_name
               FROM students st
               LEFT JOIN classes c ON c.id = st.class_id
               LEFT JOIN sections sec ON sec.id = st.section_id
               WHERE st.user_id = ?""", (session["user_id"],)).fetchone()
        data["student"] = student
        total = db.execute("SELECT COUNT(*) FROM attendance WHERE student_id = ?",
                           (student["id"],)).fetchone()[0]
        present = db.execute(
            "SELECT COUNT(*) FROM attendance WHERE student_id = ? AND status = 'Present'",
            (student["id"],)).fetchone()[0]
        data["attendance_pct"] = round(present * 100 / total, 1) if total else 0

    db.close()
    return render_template("dashboard.html", **data)


# A page only the admin can open (example of role protection)
@app.route("/admin-only-check")
@login_required
@role_required("admin")
def admin_only_check():
    return "Only admins can see this."


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)


## Step 4 — Pages (HTML templates)

In [ ]:
%%writefile school_sms/templates/base.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{% block title %}Smart School{% endblock %}</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
  <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.1/css/all.min.css">
  <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>
  <div class="toast-wrap" id="toastWrap">
    {% with messages = get_flashed_messages(with_categories=true) %}
      {% for category, message in messages %}
        <div class="toast {{ category }}">
          <i class="fa-solid {{ 'fa-circle-check' if category == 'success' else 'fa-circle-exclamation' }}"></i>
          {{ message }}
        </div>
      {% endfor %}
    {% endwith %}
  </div>

  {% block content %}{% endblock %}
  <script src="{{ url_for('static', filename='js/main.js') }}"></script>
</body>
</html>


In [ ]:
%%writefile school_sms/templates/login.html
{% extends "base.html" %}
{% block title %}Login · Smart School{% endblock %}
{% block content %}
<div class="login-page">
  <div class="login-hero">
    <div class="hero-inner">
      <div class="logo-badge"><i class="fa-solid fa-graduation-cap"></i></div>
      <h1>Smart School</h1>
      <p>One place to manage students, teachers, attendance, exams and announcements — with an AI assistant to help.</p>
      <ul>
        <li><i class="fa-solid fa-check"></i> Role-based secure access</li>
        <li><i class="fa-solid fa-check"></i> Live attendance and results</li>
        <li><i class="fa-solid fa-check"></i> Built-in AI school assistant</li>
      </ul>
    </div>
  </div>

  <div class="login-panel">
    <form class="login-card" method="POST" action="{{ url_for('login') }}">
      <h2>Welcome back 👋</h2>
      <p class="muted">Choose your role and sign in.</p>

      <div class="role-tabs" id="roleTabs">
        <button type="button" class="role-tab active" data-role="admin"><i class="fa-solid fa-user-shield"></i> Admin</button>
        <button type="button" class="role-tab" data-role="teacher"><i class="fa-solid fa-chalkboard-user"></i> Teacher</button>
        <button type="button" class="role-tab" data-role="student"><i class="fa-solid fa-user-graduate"></i> Student</button>
      </div>
      <input type="hidden" name="role" id="roleInput" value="admin">

      <label for="username">Username</label>
      <div class="input-wrap"><i class="fa-solid fa-user"></i>
        <input type="text" id="username" name="username" placeholder="Enter username" required autofocus>
      </div>

      <label for="password">Password</label>
      <div class="input-wrap"><i class="fa-solid fa-lock"></i>
        <input type="password" id="password" name="password" placeholder="Enter password" required>
      </div>

      <button type="submit" class="btn btn-primary btn-block">Sign in <i class="fa-solid fa-arrow-right"></i></button>

      <div class="demo-box">
        <strong>Demo account:</strong>
        <span id="demoText">admin / admin123</span>
        <button type="button" class="link-btn" id="fillDemo">Fill it in</button>
      </div>
    </form>
  </div>
</div>
{% endblock %}


In [ ]:
%%writefile school_sms/templates/dashboard.html
{% extends "base.html" %}
{% block title %}Dashboard · Smart School{% endblock %}
{% block content %}
<div class="app">
  <aside class="sidebar" id="sidebar">
    <div class="brand"><i class="fa-solid fa-graduation-cap"></i><span>Smart School</span></div>
    <nav>
      <a class="nav-link active" href="{{ url_for('dashboard') }}"><i class="fa-solid fa-gauge"></i> Dashboard</a>
      {% if session.role == 'admin' %}
      <a class="nav-link disabled"><i class="fa-solid fa-user-graduate"></i> Students <em>Phase 3</em></a>
      <a class="nav-link disabled"><i class="fa-solid fa-chalkboard-user"></i> Teachers <em>Phase 3</em></a>
      <a class="nav-link disabled"><i class="fa-solid fa-school"></i> Classes <em>Phase 4</em></a>
      {% endif %}
      {% if session.role in ['admin', 'teacher'] %}
      <a class="nav-link disabled"><i class="fa-solid fa-calendar-check"></i> Attendance <em>Phase 5</em></a>
      {% endif %}
      <a class="nav-link disabled"><i class="fa-solid fa-file-pen"></i> Exams &amp; Results <em>Phase 6</em></a>
      <a class="nav-link disabled"><i class="fa-solid fa-bullhorn"></i> Notices <em>Phase 7</em></a>
    </nav>
  </aside>

  <div class="main">
    <header class="topbar">
      <button class="icon-btn menu-btn" id="menuBtn" aria-label="Menu"><i class="fa-solid fa-bars"></i></button>
      <div class="topbar-title">
        <h3>Hello, {{ session.full_name }}</h3>
        <span class="muted">{{ session.role|capitalize }} account</span>
      </div>
      <div class="topbar-right">
        <div class="avatar">{{ session.full_name[0] }}</div>
        <a class="btn btn-light" href="{{ url_for('logout') }}"><i class="fa-solid fa-right-from-bracket"></i> Logout</a>
      </div>
    </header>

    <main class="content">

      {% if session.role == 'admin' %}
      <section class="cards">
        <div class="card stat"><div class="stat-icon blue"><i class="fa-solid fa-user-graduate"></i></div>
          <div><span class="muted">Total Students</span><h2>{{ total_students }}</h2></div></div>
        <div class="card stat"><div class="stat-icon purple"><i class="fa-solid fa-chalkboard-user"></i></div>
          <div><span class="muted">Total Teachers</span><h2>{{ total_teachers }}</h2></div></div>
        <div class="card stat"><div class="stat-icon green"><i class="fa-solid fa-school"></i></div>
          <div><span class="muted">Total Classes</span><h2>{{ total_classes }}</h2></div></div>
        <div class="card stat"><div class="stat-icon orange"><i class="fa-solid fa-calendar-check"></i></div>
          <div><span class="muted">Today's Attendance</span>
            <h2>{{ present_today }}<small> / {{ marked_today }}</small></h2></div></div>
      </section>

      <section class="grid-2">
        <div class="card">
          <h3 class="card-title">Upcoming Examinations</h3>
          {% if exams %}
          <table class="table">
            <thead><tr><th>Exam</th><th>Class</th><th>Starts</th><th>Ends</th></tr></thead>
            <tbody>{% for e in exams %}
              <tr><td>{{ e.name }}</td><td>{{ e.class_name }}</td><td>{{ e.start_date }}</td><td>{{ e.end_date }}</td></tr>
            {% endfor %}</tbody>
          </table>
          {% else %}<p class="muted">No upcoming exams.</p>{% endif %}
        </div>
        {% include "_notices.html" %}
      </section>

      {% elif session.role == 'teacher' %}
      <section class="grid-2">
        <div class="card">
          <h3 class="card-title">My Profile</h3>
          <ul class="info-list">
            <li><span>Name</span><b>{{ teacher.full_name }}</b></li>
            <li><span>Employee ID</span><b>{{ teacher.employee_id }}</b></li>
            <li><span>Email</span><b>{{ teacher.email }}</b></li>
            <li><span>Phone</span><b>{{ teacher.phone }}</b></li>
            <li><span>Qualification</span><b>{{ teacher.qualification }}</b></li>
          </ul>
        </div>
        <div class="card">
          <h3 class="card-title">My Classes &amp; Subjects</h3>
          <table class="table">
            <thead><tr><th>Class</th><th>Section</th><th>Subject</th></tr></thead>
            <tbody>{% for a in assignments %}
              <tr><td>{{ a.class_name }}</td><td>{{ a.section_name or '-' }}</td><td>{{ a.subject_name }}</td></tr>
            {% endfor %}</tbody>
          </table>
        </div>
      </section>
      <section>{% include "_notices.html" %}</section>

      {% else %}
      <section class="grid-2">
        <div class="card">
          <h3 class="card-title">My Profile</h3>
          <ul class="info-list">
            <li><span>Name</span><b>{{ student.full_name }}</b></li>
            <li><span>Admission No.</span><b>{{ student.admission_no }}</b></li>
            <li><span>Class</span><b>{{ student.class_name }} - {{ student.section_name }}</b></li>
            <li><span>Guardian</span><b>{{ student.guardian_name }}</b></li>
            <li><span>Contact</span><b>{{ student.phone }}</b></li>
          </ul>
        </div>
        <div class="card">
          <h3 class="card-title">My Attendance</h3>
          <div class="big-number">{{ attendance_pct }}%</div>
          <div class="progress"><div class="progress-bar" style="width: {{ attendance_pct }}%"></div></div>
          <p class="muted">Based on all days marked so far.</p>
        </div>
      </section>
      <section>{% include "_notices.html" %}</section>
      {% endif %}

    </main>
  </div>
</div>
{% endblock %}


In [ ]:
%%writefile school_sms/templates/_notices.html
<div class="card">
  <h3 class="card-title">Recent Notices</h3>
  {% if notices %}
    {% for n in notices %}
    <div class="notice">
      <div class="notice-dot"></div>
      <div>
        <b>{{ n.title }}</b>
        <p>{{ n.content }}</p>
        <span class="muted">{{ n.created_at }}</span>
      </div>
    </div>
    {% endfor %}
  {% else %}<p class="muted">No notices yet.</p>{% endif %}
</div>


## Step 5 — Design (CSS) and small JavaScript

In [ ]:
%%writefile school_sms/static/css/style.css
:root {
  --blue: #4f6bed;
  --purple: #7c4dff;
  --grad: linear-gradient(135deg, #4f6bed 0%, #7c4dff 100%);
  --bg: #f5f7fb;
  --white: #ffffff;
  --text: #1f2540;
  --muted: #7a809a;
  --border: #e6e9f2;
  --shadow: 0 6px 24px rgba(31, 37, 64, 0.07);
  --radius: 16px;
}
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: 'Inter', system-ui, sans-serif; background: var(--bg); color: var(--text); }
a { text-decoration: none; color: inherit; }
.muted { color: var(--muted); font-size: 0.88rem; }

/* ---------- buttons ---------- */
.btn { display: inline-flex; align-items: center; justify-content: center; gap: 8px; padding: 11px 18px;
  border: none; border-radius: 12px; font-weight: 600; font-size: 0.95rem; cursor: pointer;
  transition: transform .15s ease, box-shadow .15s ease; font-family: inherit; }
.btn:hover { transform: translateY(-2px); }
.btn-primary { background: var(--grad); color: #fff; box-shadow: 0 8px 20px rgba(79, 107, 237, .35); }
.btn-light { background: var(--white); color: var(--text); border: 1px solid var(--border); }
.btn-block { width: 100%; margin-top: 18px; padding: 13px; }
.icon-btn { background: var(--white); border: 1px solid var(--border); width: 42px; height: 42px;
  border-radius: 12px; cursor: pointer; font-size: 1rem; }
.link-btn { background: none; border: none; color: var(--blue); font-weight: 600; cursor: pointer; font-family: inherit; }

/* ---------- toasts ---------- */
.toast-wrap { position: fixed; top: 20px; right: 20px; z-index: 999; display: flex; flex-direction: column; gap: 10px; }
.toast { background: #fff; padding: 14px 18px; border-radius: 12px; box-shadow: var(--shadow);
  border-left: 5px solid var(--blue); font-weight: 500; animation: slideIn .35s ease; max-width: 340px; }
.toast.success { border-left-color: #22c55e; } .toast.success i { color: #22c55e; }
.toast.error { border-left-color: #ef4444; } .toast.error i { color: #ef4444; }
.toast.hide { opacity: 0; transform: translateX(30px); transition: all .4s ease; }
@keyframes slideIn { from { opacity: 0; transform: translateX(30px); } to { opacity: 1; transform: none; } }

/* ---------- login ---------- */
.login-page { display: grid; grid-template-columns: 1.1fr 1fr; min-height: 100vh; }
.login-hero { background: var(--grad); color: #fff; display: flex; align-items: center; padding: 60px; position: relative; overflow: hidden; }
.login-hero::before, .login-hero::after { content: ""; position: absolute; border-radius: 50%; background: rgba(255,255,255,.08); }
.login-hero::before { width: 380px; height: 380px; top: -120px; right: -100px; }
.login-hero::after { width: 260px; height: 260px; bottom: -80px; left: -60px; }
.hero-inner { position: relative; max-width: 460px; }
.logo-badge { width: 64px; height: 64px; background: rgba(255,255,255,.2); border-radius: 18px;
  display: grid; place-items: center; font-size: 1.8rem; margin-bottom: 24px; }
.login-hero h1 { font-size: 2.6rem; margin-bottom: 14px; }
.login-hero p { opacity: .92; line-height: 1.6; margin-bottom: 24px; }
.login-hero ul { list-style: none; display: grid; gap: 12px; }
.login-hero li i { background: rgba(255,255,255,.25); border-radius: 50%; padding: 5px; margin-right: 10px; font-size: .7rem; }
.login-panel { display: grid; place-items: center; padding: 30px; background: var(--white); }
.login-card { width: 100%; max-width: 420px; }
.login-card h2 { font-size: 1.8rem; margin-bottom: 6px; }
.login-card label { display: block; font-weight: 600; font-size: .88rem; margin: 18px 0 8px; }
.role-tabs { display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; margin: 22px 0 4px; background: var(--bg); padding: 6px; border-radius: 14px; }
.role-tab { border: none; background: transparent; padding: 10px 6px; border-radius: 10px; font-weight: 600;
  color: var(--muted); cursor: pointer; font-family: inherit; font-size: .88rem; transition: all .2s; }
.role-tab.active { background: #fff; color: var(--blue); box-shadow: var(--shadow); }
.input-wrap { position: relative; }
.input-wrap i { position: absolute; left: 14px; top: 50%; transform: translateY(-50%); color: var(--muted); }
.input-wrap input { width: 100%; padding: 13px 14px 13px 42px; border: 1.5px solid var(--border);
  border-radius: 12px; font-size: .95rem; font-family: inherit; transition: border-color .2s, box-shadow .2s; }
.input-wrap input:focus { outline: none; border-color: var(--blue); box-shadow: 0 0 0 4px rgba(79,107,237,.12); }
.demo-box { margin-top: 20px; background: #f1f3ff; padding: 12px 14px; border-radius: 12px; font-size: .85rem; }

/* ---------- app layout ---------- */
.app { display: flex; min-height: 100vh; }
.sidebar { width: 250px; background: var(--white); border-right: 1px solid var(--border); padding: 22px 16px;
  position: sticky; top: 0; height: 100vh; flex-shrink: 0; transition: transform .3s ease; }
.brand { display: flex; align-items: center; gap: 12px; font-weight: 700; font-size: 1.2rem; margin-bottom: 30px; padding: 0 8px; }
.brand i { background: var(--grad); color: #fff; padding: 10px; border-radius: 12px; font-size: 1rem; }
.sidebar nav { display: grid; gap: 6px; }
.nav-link { display: flex; align-items: center; gap: 12px; padding: 12px 14px; border-radius: 12px;
  color: var(--muted); font-weight: 500; transition: all .2s; }
.nav-link i { width: 20px; text-align: center; }
.nav-link:hover { background: var(--bg); color: var(--text); }
.nav-link.active { background: var(--grad); color: #fff; box-shadow: 0 8px 18px rgba(79,107,237,.3); }
.nav-link.disabled { opacity: .55; cursor: not-allowed; }
.nav-link em { margin-left: auto; font-size: .65rem; font-style: normal; background: var(--bg); padding: 3px 7px; border-radius: 20px; }
.main { flex: 1; min-width: 0; }
.topbar { display: flex; align-items: center; gap: 16px; padding: 16px 28px; background: var(--white);
  border-bottom: 1px solid var(--border); position: sticky; top: 0; z-index: 20; }
.topbar-title h3 { font-size: 1.1rem; }
.topbar-right { margin-left: auto; display: flex; align-items: center; gap: 14px; }
.menu-btn { display: none; }
.avatar { width: 42px; height: 42px; border-radius: 50%; background: var(--grad); color: #fff;
  display: grid; place-items: center; font-weight: 700; }
.content { padding: 28px; display: grid; gap: 24px; }

/* ---------- cards ---------- */
.cards { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 20px; }
.card { background: var(--white); border-radius: var(--radius); padding: 22px; box-shadow: var(--shadow);
  transition: transform .2s ease, box-shadow .2s ease; }
.card:hover { transform: translateY(-3px); box-shadow: 0 12px 30px rgba(31,37,64,.1); }
.card-title { margin-bottom: 16px; font-size: 1.05rem; }
.stat { display: flex; align-items: center; gap: 16px; }
.stat h2 { font-size: 1.9rem; margin-top: 2px; }
.stat h2 small { font-size: 1rem; color: var(--muted); font-weight: 500; }
.stat-icon { width: 54px; height: 54px; border-radius: 16px; display: grid; place-items: center; font-size: 1.3rem; }
.stat-icon.blue { background: #e8edff; color: var(--blue); }
.stat-icon.purple { background: #efe8ff; color: var(--purple); }
.stat-icon.green { background: #dcfce7; color: #16a34a; }
.stat-icon.orange { background: #ffedd5; color: #ea580c; }
.grid-2 { display: grid; grid-template-columns: repeat(auto-fit, minmax(340px, 1fr)); gap: 24px; }

/* ---------- tables & lists ---------- */
.table { width: 100%; border-collapse: collapse; font-size: .92rem; }
.table th { text-align: left; color: var(--muted); font-weight: 600; font-size: .8rem; text-transform: uppercase;
  letter-spacing: .04em; padding: 10px 12px; border-bottom: 1px solid var(--border); }
.table td { padding: 14px 12px; border-bottom: 1px solid var(--border); }
.table tr:hover td { background: #fafbff; }
.info-list { list-style: none; display: grid; gap: 4px; }
.info-list li { display: flex; justify-content: space-between; padding: 12px 0; border-bottom: 1px solid var(--border); }
.info-list span { color: var(--muted); }
.notice { display: flex; gap: 14px; padding: 12px 0; border-bottom: 1px solid var(--border); }
.notice:last-child { border: none; }
.notice p { color: var(--muted); font-size: .9rem; margin: 4px 0; }
.notice-dot { width: 10px; height: 10px; border-radius: 50%; background: var(--grad); margin-top: 6px; flex-shrink: 0; }
.big-number { font-size: 3rem; font-weight: 700; background: var(--grad); -webkit-background-clip: text;
  -webkit-text-fill-color: transparent; }
.progress { height: 10px; background: var(--bg); border-radius: 20px; overflow: hidden; margin: 12px 0; }
.progress-bar { height: 100%; background: var(--grad); border-radius: 20px; }

/* ---------- responsive ---------- */
@media (max-width: 900px) {
  .login-page { grid-template-columns: 1fr; }
  .login-hero { display: none; }
  .sidebar { position: fixed; z-index: 50; transform: translateX(-100%); }
  .sidebar.open { transform: none; box-shadow: var(--shadow); }
  .menu-btn { display: block; }
  .content { padding: 18px; }
  .topbar { padding: 14px 18px; }
}


In [ ]:
%%writefile school_sms/static/js/main.js
// Auto-hide toast notifications after 4 seconds
document.querySelectorAll(".toast").forEach(function (toast) {
  setTimeout(function () {
    toast.classList.add("hide");
    setTimeout(function () { toast.remove(); }, 400);
  }, 4000);
});

// Login page: role tabs + demo account helper
var demoAccounts = {
  admin: ["admin", "admin123"],
  teacher: ["teacher1", "teacher123"],
  student: ["student1", "student123"]
};
var roleInput = document.getElementById("roleInput");
if (roleInput) {
  var tabs = document.querySelectorAll(".role-tab");
  var demoText = document.getElementById("demoText");
  tabs.forEach(function (tab) {
    tab.addEventListener("click", function () {
      tabs.forEach(function (t) { t.classList.remove("active"); });
      tab.classList.add("active");
      roleInput.value = tab.dataset.role;
      var d = demoAccounts[tab.dataset.role];
      demoText.textContent = d[0] + " / " + d[1];
    });
  });
  document.getElementById("fillDemo").addEventListener("click", function () {
    var d = demoAccounts[roleInput.value];
    document.getElementById("username").value = d[0];
    document.getElementById("password").value = d[1];
  });
}

// Mobile sidebar toggle
var menuBtn = document.getElementById("menuBtn");
if (menuBtn) {
  menuBtn.addEventListener("click", function () {
    document.getElementById("sidebar").classList.toggle("open");
  });
}


## Step 6 — Start the website 🚀
Run this cell. After ~15 seconds it prints your public link. Click it to open the site.

In [ ]:
import subprocess, time, re, os

# Stop anything left over from a previous run of this cell
for name in ("flask_proc", "tunnel_proc"):
    p = globals().get(name)
    if p and p.poll() is None:
        p.terminate()

# Download cloudflared (free tunnel that gives Colab a public link) - only once
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Start Flask
flask_proc = subprocess.Popen(["python", "app.py"], cwd="school_sms",
                              stdout=open("flask.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)

# Start the tunnel
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:5000", "--no-autoupdate"],
                               stdout=subprocess.DEVNULL, stderr=open("tunnel.log", "w"))

public_url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        public_url = m.group(0)
        break

if public_url:
    print("✅ Your school website is live!")
    print("👉 Open this link:", public_url)
    print("\nDemo logins:  admin/admin123   teacher1/teacher123   student1/student123")
else:
    print("❌ Could not get a link. Run this cell again. Logs below:")
    print(open("tunnel.log").read()[-800:])
    print(open("flask.log").read()[-800:])

## 🧪 How to test Phase 1
- Log in as **admin** → you should see real numbers (students, teachers, classes, attendance), exams and notices.
- Log out, log in as **teacher1** → you should see your own profile and assigned classes.
- Log out, log in as **student1** → you should see your profile and attendance percentage.
- Try a wrong password → a red error message should appear.
- Try opening `/dashboard` after logging out → you are sent back to the login page.

## 🛠️ Something not working?
- Run the **Step 6** cell again (links expire when it stops).
- Cell shows `❌` → check the log printed under it and send it to me.
- Want to see server errors? Run `!tail -n 30 flask.log` in a new cell.

**Next:** Phase 2 (modern dashboard with charts and navigation).